In [1]:
!pip install yfinance xgboost ta joblib

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=f6a315d40d7feaf7e685e90575e9fc2bcc31868c97bec4418d349be95826147f
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [83]:
import pandas as pd
import numpy as np
import yfinance as yf
import ta
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

In [84]:
stocks = [
    "ADANIENT.NS",
    "ADANIPORTS.NS",
    "APOLLOHOSP.NS",
    "ASIANPAINT.NS",
    "AXISBANK.NS",
    "BAJAJ-AUTO.NS",
    "BAJFINANCE.NS",
    "BAJAJFINSV.NS",
    "BEL.NS",
    "BHARTIARTL.NS",
    "CIPLA.NS",
    "COALINDIA.NS",
    "DRREDDY.NS",
    "EICHERMOT.NS",
    "ETERNAL.NS",
    "GRASIM.NS",
    "HCLTECH.NS",
    "HDFCBANK.NS",
    "HDFCLIFE.NS",
    "HEROMOTOCO.NS",
    "HINDALCO.NS",
    "HINDUNILVR.NS",
    "ICICIBANK.NS",
    "INDUSINDBK.NS",
    "INFY.NS",
    "ITC.NS",
    "JIOFIN.NS",
    "JSWSTEEL.NS",
    "KOTAKBANK.NS",
    "LT.NS",
    "M&M.NS",
    "MARUTI.NS",
    "NESTLEIND.NS",
    "NTPC.NS",
    "ONGC.NS",
    "POWERGRID.NS",
    "RELIANCE.NS",
    "SBILIFE.NS",
    "SBIN.NS",
    "SHRIRAMFIN.NS",
    "SUNPHARMA.NS",
    "TATACONSUM.NS",
    "TATAMOTORS.NS",
    "TATASTEEL.NS",
    "TCS.NS",
    "TECHM.NS",
    "TITAN.NS",
    "TRENT.NS",
    "ULTRACEMCO.NS",
    "WIPRO.NS"
]

In [85]:
all_data = []

for stock in stocks:

    try:

        print(f"Downloading {stock}")

        df = yf.download(
            stock,
            start="2014-01-01",
            progress=False,
            auto_adjust=True
        )

        if len(df) == 0:
            continue

        df = df.reset_index()

        df.columns = [
            col[0] if isinstance(col, tuple)
            else col
            for col in df.columns
        ]

        df["Stock"] = stock

        all_data.append(df)

    except Exception as e:

        print(stock, e)

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TATAMOTORS.NS']: YFTzMissingError('possibly delisted; no timezone found')


In [86]:
combined_df = pd.concat(
    all_data,
    ignore_index=True
)

print(combined_df.shape)
combined_df.head()

(144366, 7)


,Date,Close,High,Low,Open,Volume,Stock
0,2014-01-01,38.012196,38.174852,37.099902,37.312063,7564701,ADANIENT.NS
1,2014-01-02,36.116886,38.012192,35.826934,37.340348,17188171,ADANIENT.NS
2,2014-01-03,35.126804,36.215898,34.582256,35.784501,11525782,ADANIENT.NS
3,2014-01-06,35.650131,36.053239,34.610542,35.063154,10660990,ADANIENT.NS
4,2014-01-07,34.221584,36.053243,34.087216,35.834010,11002957,ADANIENT.NS


In [87]:
def create_features(df):

    df = df.copy()

    close = df["Close"]
    volume = df["Volume"]

    for i in range(1, 11):

        df[f"Lag_{i}"] = close.shift(i)

    df["SMA_10"] = close.rolling(10).mean()
    df["SMA_20"] = close.rolling(20).mean()

    df["EMA_10"] = close.ewm(span=10).mean()
    df["EMA_20"] = close.ewm(span=20).mean()

    df["RSI"] = ta.momentum.RSIIndicator(close).rsi()

    macd = ta.trend.MACD(close)

    df["MACD"] = macd.macd()
    df["MACD_SIGNAL"] = macd.macd_signal()

    df["RETURN"] = close.pct_change()

    df["VOL_CHANGE"] = volume.pct_change()

    return df

In [88]:
feature_dfs = []

for stock in combined_df["Stock"].unique():

    temp = combined_df[
        combined_df["Stock"] == stock
    ].copy()

    temp = create_features(temp)

    feature_dfs.append(temp)

combined_df = pd.concat(
    feature_dfs,
    ignore_index=True
)

In [89]:
combined_df["Target"] = (
    combined_df.groupby("Stock")["Close"]
    .shift(-1)
)

In [90]:
le = LabelEncoder()

combined_df["Stock_ID"] = (
    le.fit_transform(
        combined_df["Stock"]
    )
)

In [91]:
combined_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

combined_df.dropna(
    inplace=True
)

combined_df.reset_index(
    drop=True,
    inplace=True
)

In [93]:
combined_df = combined_df.sort_values(
    by="Date"
)

In [94]:
features = [
    "Open",
    "High",
    "Low",
    "Volume",
    "SMA_10",
    "SMA_20",
    "EMA_10",
    "EMA_20",
    "RSI",
    "MACD",
    "MACD_SIGNAL",
    "RETURN",
    "VOL_CHANGE",
    "Stock_ID"
]

for i in range(1,11):

    features.append(
        f"Lag_{i}"
    )

In [95]:
split_index = int(
    len(combined_df) * 0.8
)

train_df = combined_df.iloc[
    :split_index
]

test_df = combined_df.iloc[
    split_index:
]

In [96]:
X_train = train_df[features]
y_train = train_df["Target"]

X_test = test_df[features]
y_test = test_df["Target"]

In [97]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [98]:
predictions = model.predict(
    X_test
)

In [99]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

mape = np.mean(
    np.abs(
        (y_test - predictions)
        / y_test
    )
) * 100

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)
print("MAPE:", mape)

MAE : 140.3458159259196
RMSE: 621.315706469909
R2  : 0.9493770857459065
MAPE: 2.2249172953087157


In [100]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
    model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

        Feature  Importance
2           Low    0.391509
1          High    0.231405
14        Lag_1    0.215474
15        Lag_2    0.036076
17        Lag_4    0.035272
4        SMA_10    0.018887
16        Lag_3    0.015359
20        Lag_7    0.014812
7        EMA_20    0.011868
0          Open    0.008858
6        EMA_10    0.006483
21        Lag_8    0.005624
19        Lag_6    0.005049
5        SMA_20    0.001138
18        Lag_5    0.000707
23       Lag_10    0.000596
22        Lag_9    0.000194
10  MACD_SIGNAL    0.000159
9          MACD    0.000122
8           RSI    0.000106
11       RETURN    0.000090
13     Stock_ID    0.000077
12   VOL_CHANGE    0.000072
3        Volume    0.000063


In [101]:
joblib.dump(
    model,
    "stock_model.pkl"
)

joblib.dump(
    features,
    "feature_columns.pkl"
)

joblib.dump(
    le,
    "stock_encoder.pkl"
)

['stock_encoder.pkl']

In [102]:
from google.colab import files

files.download(
    "stock_model.pkl"
)

files.download(
    "feature_columns.pkl"
)

files.download(
    "stock_encoder.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>